# Demonstração prática — YOLO26n

**Notebook reconstruído a partir dos slides 21–23** da apresentação *YOLO26 — Arquitetura, Desempenho e Aplicações* (TP558). O arquivo original `.ipynb` não está anexado nem vinculado à apresentação.

O exemplo mostrado carrega o modelo pré-treinado **YOLO26n**, obtém a imagem oficial `bus.jpg`, executa detecção com `imgsz=640` e `conf=0.25`, interpreta as caixas e salva a imagem anotada.

**Execução:** abra este arquivo no Jupyter Notebook ou no Google Colab e rode as células em ordem. Na primeira execução, são necessários acesso à internet para instalar o pacote e baixar os pesos do modelo.

Documentação oficial: https://docs.ultralytics.com/models/yolo26/

## Preparação — instalar dependências

Essa célula foi adicionada para tornar o exemplo reproduzível em um ambiente novo.

In [ ]:
%pip install -q -U ultralytics pillow requests

## Código exibido no slide — inferência e salvamento

O bloco abaixo reproduz o código mostrado no slide *"Notebook: o essencial do código"*. Ele executa as quatro etapas: carregar modelo, obter imagem, inferir e salvar.

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests, pathlib

modelo = YOLO("yolo26n.pt")
url = "https://ultralytics.com/images/bus.jpg"
caminho = pathlib.Path("imagem_teste.jpg")
caminho.write_bytes(requests.get(url,
                                 timeout=30).content)

r = modelo.predict(source=str(caminho),
                   imgsz=640, conf=0.25)[0]
print(len(r.boxes), r.speed)
Image.fromarray(r.plot()[:, :, ::-1]).save(
    "resultado_yolo26.png")

## Visualizar o resultado

A imagem anotada está salva como `resultado_yolo26.png`. A próxima célula apenas a exibe no notebook.

In [ ]:
from IPython.display import display

display(Image.open("resultado_yolo26.png"))

## Detalhes das detecções

Célula adicional para examinar **classe**, **confiança**, **caixa delimitadora** `[x1, y1, x2, y2]` em pixels e **tempo de processamento**.

O slide de resultados informa cinco detecções para essa imagem (um ônibus e quatro pessoas), mas os valores reais podem variar conforme a versão dos pesos e da biblioteca.

In [ ]:
from collections import Counter

if r.boxes is None or len(r.boxes) == 0:
    print("Nenhuma detecção acima do limiar de confiança.")
else:
    contagem = Counter()
    for i, (classe, confianca, caixa) in enumerate(
        zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist()),
        start=1,
    ):
        nome_classe = r.names[int(classe)]
        contagem[nome_classe] += 1
        coordenadas = [round(valor, 1) for valor in caixa]
        print(f"{i:02d}. {nome_classe:16s} confiança={confianca:.3f} caixa={coordenadas}")

    print("\nResumo por classe:")
    for nome_classe, quantidade in sorted(contagem.items()):
        print(f"  {nome_classe}: {quantidade}")

print("\nTempo reportado pelo modelo (ms):")
for etapa, tempo_ms in r.speed.items():
    print(f"  {etapa}: {tempo_ms:.2f} ms")

## Observação sobre NMS-free

O slide sobre a arquitetura menciona inferência **NMS-free** (`nms=False`). Porém, o exemplo de código exibido no slide do notebook **não passa** esse parâmetro. Por fidelidade ao material, mantivemos `modelo.predict(..., imgsz=640, conf=0.25)` como apresentado.

Se quiser experimentar o ramo *one-to-one* sem NMS descrito na documentação oficial, você pode alterar a chamada de inferência para:

```python
r = modelo.predict(source=str(caminho), imgsz=640, conf=0.25, nms=False)[0]
```

Isso é um **experimento opcional**, não parte do código mostrado no slide.